# ERCOT Grid Agent — Exploration Notebook

Use this notebook to explore the pipeline interactively before wiring up the CLI.
Run cells top to bottom.

In [ ]:
import sys; sys.path.insert(0, '..')
from dotenv import load_dotenv; load_dotenv('../.env')

## 1. Test inference client (Ollama)

In [ ]:
from src.inference.client import get_client

client = get_client()
print(type(client))

# Quick sanity check
response = client.complete('What is ERCOT in one sentence?')
print(response)

## 2. Streaming test

In [ ]:
for chunk in client.stream('List 3 types of ERCOT grid notices.'):
    print(chunk, end='', flush=True)
print()

## 3. Ingest mock ERCOT data

In [ ]:
from src.ingestion.ercot import fetch_mock_documents
from src.rag.store import VectorStore

store = VectorStore()
print(f'Chunks before: {store.count()}')

docs = fetch_mock_documents()
n = store.add_documents(docs)
print(f'Added {n} chunks. Total: {store.count()}')

## 4. Retrieval test

In [ ]:
question = 'What transmission constraints are active right now?'
chunks = store.query(question, top_k=3)

for i, c in enumerate(chunks, 1):
    print(f'--- Chunk {i} (distance: {c["distance"]:.4f}) ---')
    print(f'Source: {c["metadata"]["title"]}')
    print(c['text'][:200])
    print()

## 5. Full RAG answer

In [ ]:
from src.rag.agent import GridAgent

agent = GridAgent()
answer = agent.answer('What is the current grid status and are there any active outages?')
print(answer)

## 6. Switching to Baseten

Once your Truss model is deployed, just change the env var and re-run:
```python
import os
os.environ['INFERENCE_BACKEND'] = 'baseten'
os.environ['BASETEN_MODEL_ID'] = 'your_model_id'
```
Everything else is identical.